<style>
@font-face {
  font-family: 'Roboto';
  font-style: normal;
  font-weight: 600;
  src: local('Roboto Semi-Bold'), local('Roboto-SemiBold'), url('assets/Roboto-SemiBold.ttf') format('truetype');
}
</style>

<div align="center">
  <img src="assets/Jupyter_AIKit_logo.svg" width="600">
</div>

<h2 style="color:#FFFFFF; text-align:center; font-family:'Roboto', sans-serif; font-size:24px; font-weight:600; letter-spacing:0.08em;">SEMANTIC SEARCH: RAZER AIKIT WITH OPENAI API</h2>

This guide demonstrates how to perform local semantic search using the <strong>OpenAI Python SDK</strong> with an embedding model served through <strong>Razer AIKit</strong>. By leveraging AIKit’s OpenAI-compatible API, you can create fast, cloud-free search pipelines without altering standard OpenAI workflows.

By following this guide, you will learn how to:

<ul> <li>Download and serve an embedding model locally using <code>rzr-aikit</code></li> <li>Connect the OpenAI SDK to a AIKitosted local model server</li> <li>Embed a document corpus using <code>Qwen/Qwen3-Embedding-0.6B</code></li> <li>Embed user queries and compute cosine similarity for ranking</li> <li>Perform fully local semantic search in-memory</li> </ul>


<h3 style="color:#44D62C; text-align:left;">📥 1. Download a Model</h3>

Use these commands to pull a model from Hugging Face into your local environment.  
This ensures it's available for fast and offline inference.

In [ ]:
!rzr-aikit model download Qwen/Qwen3-Embedding-0.6B

Model 'Qwen/Qwen3-Embedding-0.6B' is already downloaded.


<h3 style="color:#44D62C; text-align:left;">🚀 2. Run a Model</h3>

Start a model server to perform inference locally.

When you run this command, `rzr-aikit` automatically applies optimized configuration for the selected model and your hardware (e.g., dtype, max tokens, batch size, etc.). However, you can always provide your configuration as an options.

In [ ]:
!nohup rzr-aikit model run Qwen/Qwen3-Embedding-0.6B
# Output is available in nohup.out

<h3 style="color:#44D62C; text-align:left;">📚 3. Embed the Documents</h3>

Embed a list of sample documents into vector space. These vectors will be used as the searchable index - stored fully in memory for fast lookup.

In [ ]:
from openai import OpenAI
import numpy as np

# Connect to AIKit API
client = OpenAI(
    base_url="http://localhost:8000/v1",
    api_key="rzr-aikit"
)

# Your in-memory document corpus
documents = [
    "Razer laptops deliver top-tier gaming performance.",
    "The Razer Basilisk V3 Pro is a customizable gaming mouse.",
    "Razer Synapse allows fine-tuning of RGB and macros.",
    "Our gaming headsets offer immersive THX audio.",
    "Razer keyboards are built for speed and precision."
]

# Embed each document
doc_embeddings = []
for doc in documents:
    response = client.embeddings.create(
        model="Qwen/Qwen3-Embedding-0.6B",
        input=doc
    )
    doc_embeddings.append(response.data[0].embedding)

<h3 style="color:#44D62C; text-align:left;">🧠 4. Define Semantic Search Logic</h3>

Create a reusable function to embed user queries and rank all documents by cosine similarity between vectors.

In [12]:
# Cosine similarity
def cosine_similarity(a, b):
    a = np.array(a)
    b = np.array(b)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

# Search function
def semantic_search(query, top_k=3):
    # Embed query
    response = client.embeddings.create(
        model="Qwen/Qwen3-Embedding-0.6B",
        input=query
    )
    query_embedding = response.data[0].embedding

    # Score all documents
    scores = [cosine_similarity(query_embedding, emb) for emb in doc_embeddings]
    ranked = np.argsort(scores)[::-1][:top_k]

    # Return top matches
    results = [(documents[i], scores[i]) for i in ranked]
    return results


<h3 style="color:#44D62C; text-align:left;">⚡ 5. Run Interactive Search</h3>

Enter a free-text query and return the top-matching documents from the embedded corpus - all processed locally.

In [ ]:
# Example query
query = "Which Razer product is best for FPS games?"
results = semantic_search(query)

print(f"\nQuery: {query}\n")
print("Top Matches:")
for doc, score in results:
    print(f"- ({score:.4f}) {doc}")


<h3 style="color:#44D62C; text-align:left;">🛑 6. Stop the Model</h3>

Use this command to gracefully shut down a running model service.  
This frees GPU and memory resources, and ensures clean shutdown of any background inference processes.

In [ ]:
!rzr-aikit model stop

<h3 style="color:#44D62C; text-align:left;">✅ Summary</h3>
You’ve completed a full end-to-end example of building a local semantic search pipeline using Razer AIKit and the OpenAI Python SDK.

Key highlights:

<ul> <li>Downloaded and launched a Hugging Face embedding model using <code>rzr-aikit</code></li> <li>Served the model on a local OpenAI-style endpoint</li> <li>Used the OpenAI SDK to embed documents and queries</li> <li>Ranked results by cosine similarity to find relevant matches</li> <li>Ran everything locally — fast, private, and cloud-free</li> </ul>
You’re now ready to extend this setup into a retrieval-augmented generation (RAG) system, integrate it into apps, or scale it with tools like FAISS - all powered by Razer AIKit